
# SaaS Financial Dataset Reconstruction — Rolling 12‑Month View (V5)

## Executive Context
This notebook reconstructs a **finance‑grade SaaS financial snapshot** from the RavenStack dataset.

**Key objective of V5**
Previous versions deliberately exposed pitfalls of naive churn handling. This version **fixes those issues** by strictly aligning calculations with the **dataset README semantics** and **standard SaaS CFO logic**.

The result is a dataset whose KPIs are **economically plausible** for a B2B SaaS company and suitable for **strategy consulting use**.

**Final output:** `saas_financial_snapshot.csv`



## Why a correction is required (diagnostic recap)

Earlier calculations produced:
- Gross churn above 50%
- Reactivated ARR exceeding opening ARR

Those results are **mathematically consistent** but **economically impossible** for SaaS.

**Root cause (per README):**
- `churn_events` is *event‑level*, not a permanent account status
- Many churns are **temporary**, **billing resets**, or **plan changes**
- `is_reactivation` explicitly flags non‑final churns

**V5 principle:**
> A churn event only indicates *final revenue loss* if it leads to **subscription inactivity beyond a minimum dormant period**.



## Parameters and Guardrails
The following guardrails enforce realistic SaaS economics.


In [36]:

import pandas as pd
import numpy as np
from pathlib import Path

# Paths
PATH_ACCOUNTS = Path("C:/ai-decision-intelligence-main/data/ravenstack_accounts.csv")
PATH_SUBSCRIPTIONS = Path("C:/ai-decision-intelligence-main/data/ravenstack_subscriptions.csv")
PATH_CHURN = Path("C:/ai-decision-intelligence-main/data/ravenstack_churn_events.csv")

# Rolling window (CFO standard)
ANALYSIS_DATE = None
WINDOW_DAYS = 365

# Guardrails (critical)
MIN_INACTIVE_DAYS_FOR_CHURN = 90    # churn must be followed by ≥90 days inactivity
ALLOW_SINGLE_REACTIVATION_PER_ACCOUNT = True

pd.set_option("display.float_format", "{:,.2f}".format)



## Data Loading and Window Definition


In [37]:

accounts = pd.read_csv(PATH_ACCOUNTS)
subscriptions = pd.read_csv(PATH_SUBSCRIPTIONS, parse_dates=["start_date", "end_date"])
churn_events = pd.read_csv(PATH_CHURN, parse_dates=["churn_date"])

if ANALYSIS_DATE is None:
    ANALYSIS_DATE = max(subscriptions[["start_date","end_date"]].max())

PERIOD_END = pd.Timestamp(ANALYSIS_DATE).normalize()
PERIOD_START = PERIOD_END - pd.Timedelta(days=WINDOW_DAYS)

print(f"Analysis window: {PERIOD_START.date()} → {PERIOD_END.date()}")


Analysis window: 2024-01-01 → 2024-12-31



## Account Segmentation (Executive‑Level)


In [38]:

region_map = {
    "US": "NAM", "CA": "NAM",
    "UK": "EMEA", "DE": "EMEA", "FR": "EMEA",
    "IN": "APAC", "AU": "APAC"
}
segment_map = {"Basic": "SMB", "Pro": "Mid‑Market", "Enterprise": "Enterprise"}

gross_margin_map = {"SMB":0.65,"Mid‑Market":0.75,"Enterprise":0.82}

base = accounts[["account_id","industry","country","plan_tier"]].copy()
base["region"] = base["country"].map(region_map).fillna("Other")
base["segment"] = base["plan_tier"].map(segment_map).fillna("SMB")
base["gross_margin_estimated"] = base["segment"].map(gross_margin_map)



## Step 1 — Opening ARR
Revenue at risk during the period.
Only subscriptions active at the **start of the window** are considered.


In [39]:

active_start = subscriptions[
    (subscriptions.start_date <= PERIOD_START) &
    ((subscriptions.end_date.isna()) | (subscriptions.end_date > PERIOD_START))
]

opening = (
    active_start.sort_values(["account_id","start_date"])
    .groupby("account_id", as_index=False)
    .tail(1)[["account_id","arr_amount"]]
    .rename(columns={"arr_amount":"opening_arr"})
)

print("Opening ARR accounts:", len(opening))


Opening ARR accounts: 191



## Step 2 — Valid Churned ARR (README‑Aligned)

A churn event is counted as **final revenue churn** *only if all conditions hold*:
1. The churn occurred within the analysis window
2. The account had opening ARR
3. There is **no active subscription** within `MIN_INACTIVE_DAYS_FOR_CHURN`
4. The churn is **not flagged as reactivation** (`is_reactivation = False`)

This prevents plan changes or short billing interruptions from inflating churn.


In [40]:

ce = churn_events[(churn_events.churn_date > PERIOD_START) & (churn_events.churn_date <= PERIOD_END)]

# Exclude README‑flagged reactivations
if "is_reactivation" in ce.columns:
    ce = ce[~ce.is_reactivation]

# Find next subscription start after churn
subs_after = subscriptions[["account_id","start_date"]].rename(columns={"start_date":"next_start"})

ce = ce.merge(subs_after, on="account_id", how="left")
ce["days_inactive"] = (ce["next_start"] - ce["churn_date"]).dt.days

# Valid churn definition
valid_churn = ce[(ce.days_inactive.isna()) | (ce.days_inactive >= MIN_INACTIVE_DAYS_FOR_CHURN)]

opening_idx = opening.set_index("account_id")

valid_churn["churned_arr"] = valid_churn["account_id"].map(
    lambda x: opening_idx.loc[x, "opening_arr"] if x in opening_idx.index else 0
)

churned_df = valid_churn[["account_id","churned_arr"]].drop_duplicates("account_id")
print("Valid churned accounts:", len(churned_df))


Valid churned accounts: 159



## Step 3 — New ARR (True New Logos)
Accounts without opening ARR starting their first subscription in the window.


In [41]:

new_candidates = subscriptions[(subscriptions.start_date > PERIOD_START) & (subscriptions.start_date <= PERIOD_END)]
new_candidates = new_candidates[~new_candidates.account_id.isin(opening.account_id)]

new_arr = (
    new_candidates.sort_values(["account_id","start_date"])
    .groupby("account_id", as_index=False)
    .head(1)[["account_id","arr_amount"]]
    .rename(columns={"arr_amount":"new_arr"})
)

print("New logo accounts:", len(new_arr))


New logo accounts: 309



## Step 4 — Reactivated ARR (Strict)

Reactivation is recognized **only if**:
- The account experienced **valid churn** (Step 2)
- A new subscription starts **after a minimum inactive period**
- Only the **first** valid reactivation per account is counted

This caps reactivated revenue and prevents double counting.


In [42]:

reactivation_candidates = subscriptions.merge(
    churned_df[["account_id"]], on="account_id", how="inner"
)

reactivation_candidates = reactivation_candidates[
    reactivation_candidates.start_date > PERIOD_START
]

reactivated = (
    reactivation_candidates
    .sort_values(["account_id","start_date"])
    .groupby("account_id", as_index=False)
    .head(1)[["account_id","arr_amount"]]
    .rename(columns={"arr_amount":"reactivated_arr"})
)

print("Reactivated accounts:", len(reactivated))


Reactivated accounts: 159



## Step 5 — Assemble Finance‑Grade Snapshot


In [43]:

last_sub = subscriptions.sort_values(["account_id","start_date"]).groupby("account_id", as_index=False).tail(1)
last_sub = last_sub[["account_id","arr_amount"]].rename(columns={"arr_amount":"annual_contract_value"})

final = (base
         .merge(last_sub, on="account_id", how="left")
         .merge(opening, on="account_id", how="left")
         .merge(new_arr, on="account_id", how="left")
         .merge(churned_df, on="account_id", how="left")
         .merge(reactivated, on="account_id", how="left"))

for col in ["annual_contract_value","opening_arr","new_arr","churned_arr","reactivated_arr"]:
    final[col] = final[col].fillna(0)

final["expansion_arr"] = 0
final["contraction_arr"] = 0
final["net_arr_change"] = final.new_arr + final.reactivated_arr - final.churned_arr
final["analysis_window_start"] = PERIOD_START.date()
final["analysis_window_end"] = PERIOD_END.date()

final.head()


,account_id,industry,country,plan_tier,region,segment,gross_margin_estimated,annual_contract_value,opening_arr,new_arr,churned_arr,reactivated_arr,expansion_arr,contraction_arr,net_arr_change,analysis_window_start,analysis_window_end
0,A-2e4581,EdTech,US,Basic,NAM,SMB,0.65,10032,0.00,"5,292.00",0.00,0.00,0,0,"5,292.00",2024-01-01,2024-12-31
1,A-43a9e3,FinTech,IN,Basic,APAC,SMB,0.65,10584,"4,104.00",0.00,0.00,0.00,0,0,0.00,2024-01-01,2024-12-31
2,A-0a282f,DevTools,US,Basic,NAM,SMB,0.65,1176,0.00,"45,372.00",0.00,0.00,0,0,"45,372.00",2024-01-01,2024-12-31
3,A-1f0ac7,HealthTech,UK,Basic,EMEA,SMB,0.65,14112,"14,112.00",0.00,0.00,0.00,0,0,0.00,2024-01-01,2024-12-31
4,A-ce550d,HealthTech,US,Enterprise,NAM,Enterprise,0.82,260292,0.00,"83,580.00",0.00,0.00,0,0,"83,580.00",2024-01-01,2024-12-31



## CFO‑Level Sanity Checks
These KPIs should now fall within plausible SaaS ranges.


In [44]:

opening_total = final.opening_arr.sum()
churned_total = final.churned_arr.sum()
new_total = final.new_arr.sum()
reactivated_total = final.reactivated_arr.sum()

gross_churn = churned_total / opening_total if opening_total else np.nan
grr = (opening_total - churned_total) / opening_total if opening_total else np.nan

print("Opening ARR:", f"{opening_total:,.0f}","$")
print("Churned ARR:", f"{churned_total:,.0f}","$")
print("New ARR:", f"{new_total:,.0f}","$")
print("Reactivated ARR:", f"{reactivated_total:,.0f}","$")
print("Gross churn rate:", f"{gross_churn*100:,.2f}%")
print("GRR:", f"{grr*100:,.2f}%")


Opening ARR: 4,891,680 $
Churned ARR: 1,439,928 $
New ARR: 9,281,448 $
Reactivated ARR: 3,777,048 $
Gross churn rate: 29.44%
GRR: 70.56%


In [ ]:

output = Path("C:\\ai-decision-intelligence-main\\data\\processed\\saas_financial_snapshot.csv")
if output.exists():
    output.unlink()
final.to_csv(output, index=False)
print("Exported:", output.resolve())
print("Data treatment updated")


Exported: C:\ai-decision-intelligence-main\notebooks\saas_financial_snapshot.csv
